# Промпт #26 — Self-Consistency edge case (нестабильный ответ)

**Техника:** Self-Consistency  
**Задача:** Показать как голосование спасает когда модель нестабильна  
**Сложность:** ⭐⭐⭐⭐☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion
from collections import Counter

In [5]:
# Неоднозначный отзыв без few-shot — модель будет нестабильна
prompt = """Классифицируй отзыв как ПОЗИТИВНЫЙ, НЕГАТИВНЫЙ или НЕЙТРАЛЬНЫЙ.
Дай только одно слово — метку класса, без пояснений.

Отзыв: "Качество среднее, цена завышена, но доставка быстрая и менеджер вежливый."

Ответ:"""

# Запуск без self-consistency — один запрос
print("=== БЕЗ SELF-CONSISTENCY (1 запрос) ===")
single = get_completion(prompt, temperature=0.9)
print(f"Ответ: {single}")

# Запуск с self-consistency — 7 запросов + голосование
print("\n=== С SELF-CONSISTENCY (7 запусков) ===")
answers = []
for i in range(7):
    response = get_completion(prompt, temperature=0.9).strip()
    print(f"Запуск {i+1}: {response[:50]}")
    answers.append(response)

# Голосование
print("\n=== ГОЛОСОВАНИЕ ===")
counter = Counter(answers)
print(f"Все ответы: {dict(counter)}")
winner = counter.most_common(1)[0][0]
print(f"Победитель: {winner} ({counter[winner]} из 7 запусков)")

=== БЕЗ SELF-CONSISTENCY (1 запрос) ===
Ответ: НЕЙТРАЛЬНЫЙ

=== С SELF-CONSISTENCY (7 запусков) ===
Запуск 1: НЕЙТРАЛЬНЫЙ
Запуск 2: НЕЙТРАЛЬНЫЙ
Запуск 3: НЕЙТРАЛЬНЫЙ
Запуск 4: НЕЙТРАЛЬНЫЙ
Запуск 5: НЕЙТРАЛЬНЫЙ
Запуск 6: НЕЙТРАЛЬНЫЙ
Запуск 7: НЕЙТРАЛЬНЫЙ

=== ГОЛОСОВАНИЕ ===
Все ответы: {'НЕЙТРАЛЬНЫЙ': 7}
Победитель: НЕЙТРАЛЬНЫЙ (7 из 7 запусков)


In [9]:
# Неоднозначный отзыв без few-shot — модель будет нестабильна
prompt = """Ответь одним словом: ВИНОВЕН или НЕВИНОВЕН.

Ситуация: Человек взломал базу данных больницы без разрешения. 
Он обнаружил критическую уязвимость и сообщил о ней администрации.
Благодаря этому были защищены данные 50 000 пациентов.
Он не скопировал и не использовал данные в личных целях.

Ответ:"""

# Запуск без self-consistency — один запрос
print("=== БЕЗ SELF-CONSISTENCY (1 запрос) ===")
single = get_completion(prompt, temperature=0.9)
print(f"Ответ: {single}")

# Запуск с self-consistency — 7 запросов + голосование
print("\n=== С SELF-CONSISTENCY (7 запусков) ===")
answers = []
for i in range(7):
    response = get_completion(prompt, temperature=0.9).strip()
    print(f"Запуск {i+1}: {response[:50]}")
    answers.append(response)

# Голосование
print("\n=== ГОЛОСОВАНИЕ ===")
counter = Counter(answers)
print(f"Все ответы: {dict(counter)}")
winner = counter.most_common(1)[0][0]
print(f"Победитель: {winner} ({counter[winner]} из 7 запусков)")

=== БЕЗ SELF-CONSISTENCY (1 запрос) ===
Ответ: ВИНОВЕН.

=== С SELF-CONSISTENCY (7 запусков) ===
Запуск 1: ВИНОВЕН.
Запуск 2: ВИНОВЕН.
Запуск 3: ВИНОВЕН.
Запуск 4: ВИНОВЕН.
Запуск 5: ВИНОВЕН.
Запуск 6: НЕВИНОВЕН.
Запуск 7: НЕВИНОВЕН.

=== ГОЛОСОВАНИЕ ===
Все ответы: {'ВИНОВЕН.': 5, 'НЕВИНОВЕН.': 2}
Победитель: ВИНОВЕН. (5 из 7 запусков)


## Оценка: 5/5

## Инсайт
С математическими и однозначными задачами self-consistency избыточна —
модель даёт одинаковый ответ все 7 раз. Температура не помогает когда
задача имеет один правильный ответ.

Нестабильность появилась только на этической дилемме: 5 раз ВИНОВЕН,
2 раза НЕВИНОВЕН. Голосование выбрало ВИНОВЕН как победителя.

Почему модель колебалась именно здесь:
- С точки зрения закона — взлом базы данных это преступление
- С точки зрения морали — человек защитил 50 000 пациентов
- Нет объективно правильного ответа — даже люди спорят об этом

Это и есть идеальный кейс для self-consistency: задачи где
модель реально неуверена и может пойти в разные стороны.

Главный вывод: self-consistency работает не на всех задачах.
Нужна именно та зона где модель нестабильна — субъективные,
этические, неоднозначные вопросы. На чётких задачах (математика,
факты, структурированный вывод) — просто лишние токены.